In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src')) 
from personalise_to_QRS import run_qrs_personalization
from personalise_to_Twave import run_twave_personalization
from datetime import datetime

# --- CONFIGURAZIONE DELL'ESPERIMENTO ---
subject = 'sb4301'
current_month = datetime.now().strftime('%h') 
current_year = datetime.now().strftime('%Y') 
#define unique tag to preserve history of results for same subject DEFAULT: Month_year
base_configs = {
    'exp_unique_tag': current_month + '_' + current_year,
    'verbose': True
}

# parameters for clinical ecg processing
qrs_clinical_config = {
    'clinical_ecg_filename': f'{subject}_custom_ecg.csv', #name clinical ecg file in the folder /data/clinical_data
    'source_resolution': 'coarse1500cm',
    'filtering': False, #filtering for the input ecg signal
    'low_freq_cut': 0.5, 
    'high_freq_cut': 150,
    'normalise': True, #normalising for clinical ecg signal
    'zero_align': True, 
    'frequency': 1000,  # This is only used for filtering. If you dont use 1000 Hz in any time-series in the code, the other hyper-parameters will not give the expected outcome!
    'max_len_qrs': 200,  # in ms. This is used to crop the QRS complex from the ECG signal. It should be long enough to capture the entire QRS complex, but not too long to include the T-wave.
    'lead_names': ['I', 'II', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
}
# default speed values for the fibre, normal and purkinje velocities. These are fixed during the inference, but can be changed here if desired.
# only the sheet, endo_dense and endo_sparse speeds are inferred
qrs_fixed_speed_config = {
    'fibre_speed': 0.065,  # cm/ms, Taggart et al. (2000)
    'normal_speed': 0.048,  # cm/ms, Taggart et al
    'purkinje_speed': 0.3  # cm/ms, average speed value
}
# population ranges and priors for speed and root nodes inference 
qrs_inference_speed_config = {
    'sheet_speed_range': [0.025, 0.06],  # cm
    'sheet_speed_prior': None,  # [mean, std]
    'dense_endo_speed_range': [0.1, 0.19],  # cm/ms
    'dense_endo_speed_prior': None,  # [mean, std]
    'sparse_endo_speed_range': [0.07, 0.15],  # cm/ms
    'sparse_endo_speed_prior': None,  # [mean, std]
    'nb_root_nodes_range': [3, 9],  # min/max -> 10 is computationally intractable
    'nb_root_node_prior': [6, 1],  # [mean, std]
}

# parameters for SMC-ABC inference
qrs_smc_inference_config = {
    'population_size': 120,
    'max_mcmc_steps': 50,   # This number allows for extensive exploration
    'retain_ratio': 0.5,    # proportion of samples that would match the current data in the case of N_on = 1 and all particles having the same variable switched on.
    'max_root_node_jiggle_rate': 0.1,
    'desired_discrepancy': 1.0,
    'max_process_alive_time': 24., # hours max inference time limit
    'unique_stopping_ratio': 0.5, # if only 50% of the population is unique, then terminate the inference and consider that it has converged.
    'visualisation_count':10    #minimum of 1 iteration steps to visualise the results during inference
}

qrs_final_config = {**base_configs, **qrs_clinical_config, **qrs_fixed_speed_config, **qrs_inference_speed_config, **qrs_smc_inference_config}


In [ ]:
# --- EXECUTING PHASE 1: PERSONALISE TO QRS ---

print(f"--- STARTING PHASE 1: QRS PERSONALIZATION FOR {subject} ---")

run_qrs_personalization(
    anatomy_subject_name=subject, 
    ecg_subject_name=subject, 
    **qrs_final_config
)

print(f"--- PHASE 1: QRS PERSONALIZATION COMPLETED FOR {subject} ---")


In [ ]:
# parameters for clinical ecg processing in twave personalisation
# by default are the same as qrs personalisation, but they can be changed if different
twave_clinical_config = {
    'clinical_ecg_filename': qrs_clinical_config['clinical_ecg_filename'], #name clinical ecg file in the folder /data/clinical_data
    'source_resolution': qrs_clinical_config['source_resolution'],
    'filtering': qrs_clinical_config['filtering'], #filtering for the input ecg signal
    'low_freq_cut': qrs_clinical_config['low_freq_cut'],
    'high_freq_cut': qrs_clinical_config['high_freq_cut'],
    'normalise': qrs_clinical_config['normalise'], #normalising for clinical ecg signal
    'zero_align': qrs_clinical_config['zero_align'], 
    'frequency': qrs_clinical_config['frequency'],  # This is only used for filtering. If you dont use 1000 Hz in any time-series in the code, the other hyper-parameters will not give the expected outcome!
    'max_len_qrs': qrs_clinical_config['max_len_qrs'],  # in ms. This is used to crop the QRS complex from the ECG signal. It should be long enough to capture the entire QRS complex, but not too long to include the T-wave.
    'max_len_st': 300,  # in ms. This is used to crop the ST segment from the ECG signal. It should be long enough to capture the entire ST segment, but not too long to include the T-wave.
    'lead_names': qrs_clinical_config['lead_names']
}

twave_inference_ranges_config = {
    'apd_exploration_margin': 80,
    'apd_max_prior': None,  # [mean, std]
    'apd_min_prior': None,  # [mean, std]
    'g_vc_ab_range': [-1, 1], # min-max
    'g_vc_ab_prior': None,  # [mean, std]
    'g_vc_aprt_range': [-1, 1], # min-max
    'g_vc_aprt_prior': None,  # [mean, std]
    'g_vc_rvlv_range': [-1, 1], # min-max
    'g_vc_rvlv_prior': None,  # [mean, std]
    'g_vc_tm_range': [-1, 1], # min-max
    'g_vc_tm_prior': None,  # [mean, std]
}

twave_smc_inference_config = {
    'population_size': 120,
    'max_mcmc_steps': 50,   # This number allows for extensive exploration
    'retain_ratio': 0.5,    # proportion of samples that would match the current data in the case of N_on = 1 and all particles having the same variable switched on.
    'unique_stopping_ratio': 0.5, # if only 50% of the population is unique, then terminate the inference and consider that it has converged.
    'max_root_node_jiggle_rate': 0.1,
    'desired_discrepancy': 1.0,
    'max_process_alive_time': 24., # hours max inference time limit
    'visualisation_count':10,    #minimum of 1 iteration steps to visualise the results during inference
}
twave_final_config = {**base_configs, **twave_clinical_config, **twave_inference_ranges_config, **twave_smc_inference_config}


In [ ]:
# --- EXECUTING PHASE 2: PERSONALISE TO TWAVE ---

print(f"--- STARTING PHASE 2: TWAVE PERSONALIZATION FOR {subject} ---")

run_twave_personalization(
    anatomy_subject_name=subject, 
    ecg_subject_name=subject, 
    **twave_final_config
)

print(f"--- PHASE 2: TWAVE PERSONALIZATION COMPLETED FOR {subject} ---")
